In [ ]:
import os # Added for os.getcwd()
'''
# Removed: !git clone https://github.com/git (this was incorrect)
# Removed: %cd /content/IndicTrans2/huggingface_interface (this path did not exist)

!python3 -m pip install nltk sacremoses pandas regex mock transformers==4.53.2 mosestokenizer
!python3 -c "import nltk; nltk.download('punkt')"
!python3 -m pip install bitsandbytes scipy accelerate datasets
!python3 -m pip install sentencepiece

!git clone https://github.com/VarunGumma/IndicTransToolkit.git
import sys

# The toolkit is cloned to /content/IndicTransToolkit, so the path should reflect that.
sys.path.append("/content/IndicTransToolkit/IndicTransToolkit")

%cd /content/IndicTransToolkit
!python3 -m pip install --editable ./
%cd ..

print(f"Current working directory after setup: {os.getcwd()}")

# Restart runtime after this cell
This all are the codes required to run this notebook, but as it was done in google colab so the syntax are quite different.
'''

In [ ]:
import os
print(os.listdir("/content"))

In [ ]:
import json

with open("/content/mapping.json", "r", encoding="utf-8") as f:
    mapping = json.load(f)

In [ ]:
import re

def apply_mapping(sentences, mapping):
    mapped_sentences = []

    for sentence in sentences:
        new_sentence = sentence

        for key, value in mapping.items():
            # Case-insensitive replace
            pattern = r'\b' + re.escape(key) + r'\b'
            new_sentence = re.sub(pattern, value, new_sentence, flags=re.IGNORECASE)

        mapped_sentences.append(new_sentence)

    return mapped_sentences

Total comments: 4414


In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig, AutoTokenizer

import sys
# Corrected toolkit_path based on actual installation location
toolkit_path = "/content/IndicTransToolkit/IndicTransToolkit"
if toolkit_path not in sys.path:
    sys.path.append(toolkit_path)

from processor import IndicProcessor

BATCH_SIZE = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
quantization = None


# =========================
# Initialize model & tokenizer
# =========================
def initialize_model_and_tokenizer(ckpt_dir, quantization):
    if quantization == "4-bit":
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    elif quantization == "8-bit":
        qconfig = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_use_double_quant=True,
            bnb_8bit_compute_dtype=torch.bfloat16,
        )
    else:
        qconfig = None

    tokenizer = AutoTokenizer.from_pretrained(ckpt_dir, trust_remote_code=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        ckpt_dir,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconfig,
    )

    if qconfig is None:
        model = model.to(DEVICE)
        if DEVICE == "cuda":
            model.half()

    model.eval()
    return tokenizer, model


# =========================
# Batch translation
# =========================
def batch_translate(input_sentences, src_lang, tgt_lang, model, tokenizer, ip):
    translations = []
    for i in range(0, len(input_sentences), BATCH_SIZE):
        batch = input_sentences[i : i + BATCH_SIZE]

        batch = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)

        inputs = tokenizer(
            batch,
            truncation=True,
            padding="longest",
            return_tensors="pt",
            return_attention_mask=True,
        ).to(DEVICE)

        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                use_cache=True,
                min_length=0,
                max_length=256,
                num_beams=5,
                num_return_sequences=1,
            )

        generated_tokens = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        translations += ip.postprocess_batch(generated_tokens, lang=tgt_lang)

        del inputs
        torch.cuda.empty_cache()

    return translations


# =========================
# HuggingFace Login
# =========================
import os 
from huggingface_hub import login
HF_TOKENS = os.getenv("HF_TOKENS")
login(HF_TOKENS)


# =========================
# Initialize model/tokenizer & processor
# =========================
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, quantization)
ip = IndicProcessor(inference=True)


# =========================
# English Comments Only
# =========================


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 227.53it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForTokenClassification LOAD REPORT from: Davlan/xlm-roberta-base-ner-hrl
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'ORG', 'score': np.float32(0.99756145), 'word': 'कांग्रेस', 'start': 0, 'end': 8}]
[]


In [ ]:
en_sents = [

    "He has many old books, which he inherited from his ancestors.",
    "My name is Pooja.",
    "i support rsp",
    "puspa kamal dahal is the mastermind",
    "NC has been really cruel to the protestors",
    "uml is not someone i support",
    "Balen sah is pm",
    "Harka samphang is mayor",
    "K.P Oli",
    "He cooks delicious food.",
    "Love this! Keep up the great work 😊",
    "Interesting take, never thought about it that way.",
    "Thanks for sharing this, very helpful!",
    "Hmm, I need to think about this a bit more.",
    "Not sure if I agree, but it’s an interesting perspective.",
    "This seems useful, I’ll keep it in mind.",
    "Wow, this is really insightful!",
    "Amazing content, totally made my day.",
    "This is exactly what I was looking for, thank you!",
    "Super helpful post, appreciate it!",
    "This made me laugh way more than it should!",
    "Can someone explain this in simpler terms?",
    "I’m curious, how did you come up with this idea?",
    "Does this work for everyone or just certain cases?",
    "I’d love to see more examples like this.",
    "What’s the source of this info?",
"Honestly after watching this full interview I still feel like our leaders talk a lot but say very little. Every election they promise development jobs better roads and hospitals but after winning everything becomes silent. Nepal has so much potential yet we are still struggling with basic infrastructure. As a citizen it feels frustrating and tiring to hear the same speeches again and again.",
"I respect the journalist for asking direct questions instead of only praising the politician. But the answers felt very diplomatic and carefully managed. Youth are leaving the country every day and still we hear the same political arguments. We need practical solutions not emotional speeches.",
"To be honest I do not fully agree with everything said in this video but some points were valid. Corruption is still a major issue and everyone knows it. It is not about one party anymore it is about the entire system. Until accountability becomes serious I do not see major change happening.",
"Why are people fighting in the comment section about parties when the real problem is governance and transparency? No matter who comes to power if the mindset does not change the result will be the same. Nepal deserves better leadership and long term planning.",
"I watched the entire discussion and I appreciate the calm tone. However I feel like important questions about economic reform were avoided. Inflation is rising job opportunities are limited and small businesses are struggling. These issues need direct answers not political comparisons.",
"Sometimes I feel like our politicians are disconnected from the ground reality. They speak about big visions and foreign investments but what about local employment and rural development? Many villages still lack proper roads and healthcare facilities.",
"This interview was interesting but I am still confused about the actual plan moving forward. There was a lot of blame on previous governments but very little clarity on future strategy. We want a roadmap not excuses.",
"As a young voter I honestly feel disappointed. Every leader says youth are the future of the country but policies do not support entrepreneurship or innovation. Many talented people are forced to go abroad. That is a serious loss for Nepal.",
"I appreciate that the discussion stayed respectful but at the end of the day respect alone is not enough. We need transparency measurable goals and regular updates on progress. Otherwise it just becomes another political talk show.",
"After listening carefully I think some reforms are possible but only if there is strong political will. Nepal has already wasted too much time in internal conflicts and power struggles. It is time to focus on development and stability."

]

src_lang, tgt_lang = "eng_Latn", "npi_Deva"
# Apply mapping BEFORE translation
mapped_sentences = apply_mapping(en_sents, mapping)

# Then translate
hi_translations = batch_translate(mapped_sentences, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)


# =========================
# Output
# =========================
print(f"\n{src_lang} - {tgt_lang}")
for input_sentence, translation in zip(en_sents, hi_translations):
    print(f"{src_lang}: {input_sentence}")
    print(f"{tgt_lang}: {translation}")

NER extraction complete
Unique raw entities: 530


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Sample entities:
['सिंहदरबार', 'बालेन', 'सुदन गुरुङ', 'ह', 'र्के', 'ओली', 'देउवा', 'प्रचण्ड', 'बाल', 'रवि नमि', 'हर्क सम्', 'सु', 'दन गुरुङ', 'झा', 'झापा', 'कुकुर', 'के पी ओली', 'केपी शर्मा ओली', 'केपी ओलीको', 'नेपाल']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Folder in Google Drive where you want to save the transliteration model
save_dir = "/content/drive/MyDrive/models/transliteration_model"

# Save the model and tokenizer
en_indic_model.save_pretrained(save_dir)
en_indic_tokenizer.save_pretrained(save_dir)

print("✅ Transliteration model and tokenizer saved to:", save_dir)

c:\Users\ACER\OneDrive\Desktop\NLP project\venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ACER\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 263.65

In [13]:
clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=0.35,
    metric="cosine",
    linkage="average"
)

cluster_ids = clustering.fit_predict(entity_embeddings)


In [14]:
cluster_map = defaultdict(list)

for entity, cluster_id in zip(unique_entities, cluster_ids):
    cluster_map[cluster_id].append(entity)

print("Total clusters:", len(cluster_map))


Total clusters: 71


In [15]:
canonical_map = {}

for cluster_id, variants in cluster_map.items():
    canonical = max(variants, key=lambda v: entity_frequency[v])

    for v in variants:
        canonical_map[v] = canonical


In [16]:
normalized_comment_entities = []

for ents in comment_entities:
    normalized = []

    for e in ents:
        canonical = canonical_map.get(e["word"], e["word"])

        normalized.append({
            "canonical": canonical,
            "label": e["label"],
            "score": e["score"]
        })

    normalized_comment_entities.append(normalized)


In [17]:
dropdown_targets = sorted(list({
    e["canonical"]
    for comment in normalized_comment_entities
    for e in comment
}))

print("\nFINAL DROPDOWN TARGETS:")
print(dropdown_targets)
print("Total dropdown targets:", len(dropdown_targets))



FINAL DROPDOWN TARGETS:
['आन्तरिक राजस्व विभाग', 'उद्योग', 'एमसीसी', 'एमाले', 'एसजीबी', 'एस्पालि', 'ओली', 'कांग्रेस', 'काङ्ग्रेस', 'किरण', 'कृषि', 'के पी ओली', 'के पी चोर', 'केदार न्यौपाने', 'केवलकार', 'गिरि बन्द', 'चन्द्र सर', 'चाटु मिडिया', 'चिन तिब्बत', 'चिनकाजी महर्जन', 'चीन', 'जिन्दाबाद', 'जु', 'ज्ञान', 'डिल्ली बजार', 'तीन', 'दक्षिण कोरिया', 'दरबारमार्ग कन्सर्ट', 'दिल भुषण पाठक', 'दुर्गा प्रसाई', 'देउबा', 'धादिंग ढोला', 'ध्रुब राठी', 'निर्मला', 'नेपाल', 'नेपाल राष्ट्र बैंकले', 'पशुपतिनाथ', 'पिएचडी', 'बंगलादेश', 'बम', 'बल', 'बहादुर', 'बाल', 'बालेन', 'बि', 'बुढीगण्डकी जलविद्युत आयोजना', 'भारत', 'माओवादी', 'माफ', 'मिटरब्याजी', 'मोदी', 'रमेश लेखक', 'रवि दाई जय घण्टी', 'रवि लामिछाने', 'राजा', 'राम', 'रावण', 'रास्वपा', 'लनाथ बादलको', 'विद्युत प्राधिकरण', 'श्रीलंका', 'सत्य', 'सम्पत्ति शुद्धीकरण', 'सर्वोच्च', 'साउदी', 'सिंहदरबार', 'सुन', 'स्विच बै', 'स्विजरल्याण्ड', 'हरे', 'हर्के माचिक्नी']
Total dropdown targets: 71
